# 🧱 Structured Outputs

Constrain decoding so the model literally cannot emit invalid JSON.

> ▶︎ In Colab: **Runtime → Run all** — this notebook runs top to bottom with no setup.

In [ ]:
%pip install -q numpy

In [ ]:
# Constrained decoding masks tokens that would break the format.
# Toy version: only allow characters valid in JSON given what's been written.
def allowed_next(partial):
    if partial == '':
        return ['{']
    if partial == '{':
        return ['"']                 # a key must start with a quote
    if partial.endswith('}'):
        return []                      # done
    return ['"', ':', '}', 'a', 'b', '1']

print('start -> allowed:', allowed_next(''))
print('{     -> allowed:', allowed_next('{'))

## Try it

Drive the masked generation: at each step pick only from the allowed set. The output is valid by construction, not by luck.

In [ ]:
import json
out = ''
for ch in '{"x":1}':            # a 'model' that proposes these chars
    if ch in allowed_next(out) or allowed_next(out) == []:
        out += ch
print('generated:', out)
print('parses cleanly:', json.loads(out))

## Takeaway

- Constrained decoding guarantees parseable output.
- It's what makes LLMs safe to wire into real software and tool calls.

## 🚀 Your move

Build: write a tiny constrained generator that, at each step, only allows tokens valid in JSON given what's been emitted so far. Even a toy version shows why 'the model literally can't produce broken JSON' beats hoping it won't.